# Milestone 1 crawler on Colab (CPU)

One machine crawls the **whole** set (`id_ranges.csv`, D8). Put `S2_API_KEY` in Colab **Secrets** (never in the notebook).

**Chạy khi rảnh, dừng khi bận — không mất dữ liệu** if you follow this loop every session:

1. Setup (clone, mount Drive, secrets) — once per session
2. **Restore** Drive → `/content/data` — always before crawl
3. **Crawl** — same command every time; manifest skips finished IDs (auto-backup every 1000 IDs to Drive when env is set)
4. When you stop: **Runtime → Interrupt**, then run **Backup** if you want a final rsync
5. Next session: repeat from step 1

Data lives on Colab local disk while crawling (SQLite + Drive FUSE is unsafe). Backup copies the full `data/` folder to `MyDrive/arxiv-data/`.

In [ ]:
import os

if not os.path.isdir("/content/arxiv-pipeline"):
    !git clone https://github.com/GiaaPhuc/arxiv-pipeline.git
else:
    !git -C /content/arxiv-pipeline pull --ff-only
%cd /content/arxiv-pipeline/src
!pip install -q -r requirements.txt

In [ ]:
import os

from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=True)
os.environ["S2_API_KEY"] = userdata.get("S2_API_KEY")
os.environ["CRAWLER_CONTACT"] = "your-email@example.com"  # User-Agent contact
os.environ["CRAWLER_DATA_DIR"] = "/content/data"
os.environ["CRAWLER_BACKUP_DIR"] = "/content/drive/MyDrive/arxiv-data"
os.environ["CRAWLER_BACKUP_EVERY_IDS"] = "1000"
os.environ["CRAWLER_ARXIV_MIN_INTERVAL"] = "3.5"  # Colab: small margin above 3 s

In [ ]:
# Plan only: no network.
!python run_crawl.py --ranges id_ranges.csv --dry-run

In [ ]:
# Restore from Drive — run EVERY new session (safe on first run if Drive is empty).
!mkdir -p /content/data /content/drive/MyDrive/arxiv-data
!rsync -a /content/drive/MyDrive/arxiv-data/ /content/data/

In [ ]:
# Crawl: re-run the SAME active line after every restore to resume.
# Exit code 3 = arXiv still rate-limiting after waits — wait 30–60 min, Backup, try again later.
# First session — pilot 30 IDs (switch to full crawl after pilot OK):
!python run_crawl.py --ranges id_ranges.csv --limit 30
# !python run_crawl.py --ranges id_ranges.csv

### Dừng an toàn (trước khi đóng tab / tắt máy / hết giờ Colab)

1. **Runtime → Interrupt execution** (or wait until you want to stop — Ctrl+C is OK; manifest keeps progress).
2. Run the **Backup** cell below and wait until `rsync` finishes.
3. You can disconnect; next session: **Restore** then the same crawl command.

Do **not** re-run **Restore** in the same session after crawling unless you already backed up — it would overwrite local data with the last Drive snapshot.

In [ ]:
# Backup — run after Interrupt and before closing the session.
!mkdir -p /content/drive/MyDrive/arxiv-data
!rsync -a /content/data/ /content/drive/MyDrive/arxiv-data/

In [ ]:
# ONLY when crawl finished (no exit 3 / no pending pairs) — not after a partial run:
# !python validate_corpus.py --data-dir $CRAWLER_DATA_DIR
# !python export_data.py --data-dir $CRAWLER_DATA_DIR